In [8]:
# transformer model

import os, gc, json, codecs, re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, accuracy_score
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from preprocessing_class import Preprocessing, load_pres, load_movies

# CUDA memory frag
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Config for run
DATASET = "movies"

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
GRADIENT_ACCUMULATION_STEPS = 2   # to accumulate batches

# load data
alltxt, alllabs = None, None

print("Loading DATASET", DATASET)

if DATASET == "pres": 
    alltxt, alllabs = load_pres("Dataset/corpus.tache1.learn.utf8")
    alltxt, alllabs = np.array(alltxt), np.array(alllabs)
    # Convert labels: 1 -> 0, -1 -> 1
    alllabs = np.where(alllabs == 1, 0, 1)
else:
    alltxt, alllabs = load_movies("./Dataset/movies1000/")
    alltxt, alllabs = np.array(alltxt), np.array(alllabs)
print(f"dataset size: {len(alltxt)}")

# weigths
#val, counts = np.unique(alllabs, return_counts=True)
#class_weights = 1 / counts

#print(f"weights: {class_weights}")

# splits 
X_train, X_test, y_train, y_test = train_test_split(
    alltxt, alllabs,
    test_size=TEST_SIZE,
    stratify=alllabs,
    random_state=RANDOM_STATE
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=VAL_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE
)

print(f" Train: {len(X_train_sub)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Convert to tensors
y_train_tens = torch.tensor(y_train_sub, dtype=torch.long)
y_val_tens = torch.tensor(y_val, dtype=torch.long)
y_test_tens = torch.tensor(y_test, dtype=torch.long)

Loading DATASET movies
dataset size: 2000
 Train: 1280, Val: 320, Test: 400


In [9]:
def clear_gpu_memory_completely():
    """Aggressively clear ALL GPU memory between runs"""
    import gc
    import torch
    
    # 1. Delete ALL common GPU objects
    for name in dir():
        obj = globals().get(name)
        if isinstance(obj, torch.Tensor) and obj.is_cuda:
            del globals()[name]
        elif hasattr(obj, 'parameters') and hasattr(obj, 'to'):  # Models
            try:
                obj.cpu()
                del obj
            except:
                pass
    
    # 2. Force garbage collection
    gc.collect()
    
    # 3. Clear PyTorch cache (twice for stubborn memory)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.empty_cache()  # Double clear
    
    # 4. Reset CUDA context (nuclear option)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    
    print(f"GPU memory cleared. Allocated: {torch.cuda.memory_allocated()/1e6:.0f}MB, Cached: {torch.cuda.memory_reserved()/1e6:.0f}MB")

In [ ]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

MAX_LENGTH = 128            
BATCH_SIZE = 8            
LEARNING_RATE = 2e-5
FREEZE_TRANSFORMER = False # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "FacebookAI/roberta-base"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


<>:265: SyntaxWarning: invalid escape sequence '\S'
<>:265: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_5559/1956235158.py:265: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")
/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU memory available: 6.03 GB
Loading DATASET movies
dataset size: 2000
weights: [0.001 0.001]
 Train: 1280, Val: 320, Test: 400


Tokenizing train
Tokenizing validation
Tokenizing test

Model used (FacebookAI/roberta-base)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4262.94it/s]
RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training epochs :  20

Epoch 1/20
Model improved
train loss 0.6938 val loss 0.6517 | train f1 0.5223 val f1 0.5971 | train auc 0.5304 val auc 0.7168

Epoch 2/20
Model improved
train loss 0.5909 val loss 0.5629 | train f1 0.6989 val f1 0.6833 | train auc 0.7531 val auc 0.7893

Epoch 3/20
Model improved
train loss 0.4883 val loss 0.6420 | train f1 0.7994 val f1 0.7244 | train auc 0.8470 val auc 0.7745

Epoch 4/20
patience 1/5
train loss 0.3967 val loss 0.6412 | train f1 0.8402 val f1 0.7157 | train auc 0.9053 val auc 0.8191

Epoch 5/20
patience 2/5
train loss 0.4086 val loss 0.7288 | train f1 0.8723 val f1 0.6810 | train auc 0.9104 val auc 0.8071

Epoch 6/20
Model improved
train loss 0.3104 val loss 0.7659 | train f1 0.8845 val f1 0.7325 | train auc 0.9424 val auc 0.8209

Epoch 7/20
Model improved
train loss 0.2765 val loss 1.0575 | train f1 0.9211 val f1 0.7596 | train auc 0.9552 val auc 0.8309

Epoch 8/20
Model improved
train loss 0.1880 val loss 1.1486 | train f1 0.9509 val f1 0.7630 

In [10]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

MAX_LENGTH = 256            
BATCH_SIZE = 8            
LEARNING_RATE = 2e-5
FREEZE_TRANSFORMER = False # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "FacebookAI/roberta-base"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


<>:199: SyntaxWarning: invalid escape sequence '\S'
<>:199: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_29181/720361441.py:199: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


GPU memory cleared. Allocated: 451MB, Cached: 487MB
Using device: cuda
GPU memory available: 6.03 GB


Tokenizing train
Tokenizing validation
Tokenizing test

Model used (FacebookAI/roberta-base)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8793.46it/s]
RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training epochs :  20

Epoch 1/20
Model improved
train loss 0.7048 val loss 0.6926 | train f1 0.6646 val f1 0.6667 | train auc 0.5343 val auc 0.7547

Epoch 2/20
Model improved
train loss 0.6270 val loss 0.4994 | train f1 0.6233 val f1 0.7850 | train auc 0.6937 val auc 0.8743

Epoch 3/20
patience 1/5
train loss 0.4693 val loss 0.5897 | train f1 0.8272 val f1 0.7407 | train auc 0.8628 val auc 0.8727

Epoch 4/20
Model improved
train loss 0.3981 val loss 0.6473 | train f1 0.8625 val f1 0.8190 | train auc 0.9106 val auc 0.8980

Epoch 5/20
patience 1/5
train loss 0.3474 val loss 0.7358 | train f1 0.8980 val f1 0.8182 | train auc 0.9310 val auc 0.8793

Epoch 6/20
patience 2/5
train loss 0.2665 val loss 0.5948 | train f1 0.9180 val f1 0.8165 | train auc 0.9635 val auc 0.9084

Epoch 7/20
patience 3/5
train loss 0.2384 val loss 0.7984 | train f1 0.9321 val f1 0.8173 | train auc 0.9751 val auc 0.9061

Epoch 8/20
patience 4/5
train loss 0.1538 val loss 0.8610 | train f1 0.9628 val f1 0.8173 | trai

In [11]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

MAX_LENGTH = 512            
BATCH_SIZE = 8            
LEARNING_RATE = 2e-5
FREEZE_TRANSFORMER = False # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "FacebookAI/roberta-base"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


GPU memory cleared. Allocated: 2446MB, Cached: 2647MB
Using device: cuda
GPU memory available: 6.03 GB


<>:199: SyntaxWarning: invalid escape sequence '\S'
<>:199: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_29181/728945141.py:199: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


Tokenizing train
Tokenizing validation
Tokenizing test

Model used (FacebookAI/roberta-base)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8770.04it/s]
RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training epochs :  20

Epoch 1/20
Model improved
train loss 0.5525 val loss 0.3540 | train f1 0.6492 val f1 0.8397 | train auc 0.7896 val auc 0.9264

Epoch 2/20
Model improved
train loss 0.3399 val loss 0.6044 | train f1 0.8808 val f1 0.8603 | train auc 0.9298 val auc 0.9301

Epoch 3/20
patience 1/5
train loss 0.3211 val loss 0.5752 | train f1 0.9080 val f1 0.8571 | train auc 0.9514 val auc 0.9426

Epoch 4/20
patience 2/5
train loss 0.1958 val loss 0.5976 | train f1 0.9491 val f1 0.8435 | train auc 0.9699 val auc 0.9540

Epoch 5/20
Model improved
train loss 0.1791 val loss 0.7385 | train f1 0.9573 val f1 0.8732 | train auc 0.9790 val auc 0.9426

Epoch 6/20
Model improved
train loss 0.1403 val loss 0.5181 | train f1 0.9678 val f1 0.8743 | train auc 0.9840 val auc 0.9498

Epoch 7/20
Model improved
train loss 0.1556 val loss 0.5366 | train f1 0.9651 val f1 0.8843 | train auc 0.9834 val auc 0.9537

Epoch 8/20
Model improved
train loss 0.0938 val loss 0.6191 | train f1 0.9717 val f1 0.8917 

In [12]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

MAX_LENGTH = 256            
BATCH_SIZE = 8            
LEARNING_RATE = 2e-5
FREEZE_TRANSFORMER = True # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "FacebookAI/roberta-base"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


GPU memory cleared. Allocated: 2446MB, Cached: 2793MB
Using device: cuda
GPU memory available: 6.03 GB


<>:199: SyntaxWarning: invalid escape sequence '\S'
<>:199: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_29181/3461672410.py:199: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


Tokenizing train
Tokenizing validation
Tokenizing test

Model used (FacebookAI/roberta-base)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8967.73it/s]
RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base roberta frozen. Classifier head is still trainable.
Training epochs :  20

Epoch 1/20
Model improved
train loss 0.6992 val loss 0.6921 | train f1 0.6448 val f1 0.6385 | train auc 0.5261 val auc 0.6131

Epoch 2/20
Model improved
train loss 0.6953 val loss 0.6912 | train f1 0.1911 val f1 0.6539 | train auc 0.5039 val auc 0.6611

Epoch 3/20
patience 1/5
train loss 0.6900 val loss 0.6915 | train f1 0.5826 val f1 0.0000 | train auc 0.5498 val auc 0.6672

Epoch 4/20
Model improved
train loss 0.6928 val loss 0.6902 | train f1 0.5328 val f1 0.6667 | train auc 0.5283 val auc 0.6896

Epoch 5/20
patience 1/5
train loss 0.6881 val loss 0.6899 | train f1 0.5906 val f1 0.6667 | train auc 0.5666 val auc 0.6970

Epoch 6/20
patience 2/5
train loss 0.6874 val loss 0.6886 | train f1 0.5613 val f1 0.0606 | train auc 0.5829 val auc 0.6953

Epoch 7/20
Model improved
train loss 0.6869 val loss 0.6875 | train f1 0.4979 val f1 0.6914 | train auc 0.5807 val auc 0.7024

Epoch 8/20
patience 1/5
train loss 0.

In [13]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

#smoothing
MAX_LENGTH = 256            
BATCH_SIZE = 8            
LEARNING_RATE = 2e-6
FREEZE_TRANSFORMER = True # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "FacebookAI/roberta-base"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


<>:200: SyntaxWarning: invalid escape sequence '\S'
<>:200: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_29181/1831651261.py:200: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


GPU memory cleared. Allocated: 1453MB, Cached: 1850MB
Using device: cuda
GPU memory available: 6.03 GB
Tokenizing train
Tokenizing validation
Tokenizing test

Model used (FacebookAI/roberta-base)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9541.54it/s]
RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base roberta frozen. Classifier head is still trainable.
Training epochs :  20

Epoch 1/20
Model improved
train loss 0.6938 val loss 0.6932 | train f1 0.5591 val f1 0.6667 | train auc 0.5036 val auc 0.5089

Epoch 2/20
patience 1/5
train loss 0.6933 val loss 0.6931 | train f1 0.5955 val f1 0.6667 | train auc 0.5178 val auc 0.5191

Epoch 3/20
patience 2/5
train loss 0.6930 val loss 0.6930 | train f1 0.5590 val f1 0.6567 | train auc 0.5118 val auc 0.5294

Epoch 4/20
patience 3/5
train loss 0.6951 val loss 0.6929 | train f1 0.5203 val f1 0.6450 | train auc 0.4892 val auc 0.5369

Epoch 5/20
patience 4/5
train loss 0.6920 val loss 0.6928 | train f1 0.5501 val f1 0.6267 | train auc 0.5305 val auc 0.5464

Epoch 6/20
patience 5/5
train loss 0.6946 val loss 0.6928 | train f1 0.5401 val f1 0.6395 | train auc 0.4978 val auc 0.5552
\Stopping early at epoch 6

Best validation F1 obtained: 0.6667

Test | loss 0.6930 | f1 0.6644 | prec 0.4987 | rec 0.9950 | auc 0.5254 | ap 0.5679

Saved to movies_tran

In [14]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

#bert
MAX_LENGTH = 512            
BATCH_SIZE = 16          
LEARNING_RATE = 2e-6
FREEZE_TRANSFORMER = True # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "bert-base-cased"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


<>:200: SyntaxWarning: invalid escape sequence '\S'
<>:200: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_29181/174485906.py:200: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


GPU memory cleared. Allocated: 1453MB, Cached: 1535MB
Using device: cuda
GPU memory available: 6.03 GB
Tokenizing train
Tokenizing validation
Tokenizing test

Model used (bert-base-cased)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9066.94it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

Base bert frozen. Classifier head is still trainable.
Training epochs :  20

Epoch 1/20
Model improved
train loss 0.7345 val loss 0.7266 | train f1 0.6635 val f1 0.6667 | train auc 0.4872 val auc 0.5209

Epoch 2/20
patience 1/5
train loss 0.7306 val loss 0.7193 | train f1 0.6642 val f1 0.6667 | train auc 0.4780 val auc 0.5223

Epoch 3/20
patience 2/5
train loss 0.7174 val loss 0.7139 | train f1 0.6607 val f1 0.6667 | train auc 0.5216 val auc 0.5236

Epoch 4/20
patience 3/5
train loss 0.7157 val loss 0.7101 | train f1 0.6628 val f1 0.6667 | train auc 0.5020 val auc 0.5241

Epoch 5/20
patience 4/5
train loss 0.7115 val loss 0.7053 | train f1 0.6566 val f1 0.6667 | train auc 0.5031 val auc 0.5259

Epoch 6/20
patience 5/5
train loss 0.7076 val loss 0.7020 | train f1 0.6456 val f1 0.6639 | train auc 0.5019 val auc 0.5272
\Stopping early at epoch 6

Best validation F1 obtained: 0.6667

Test | loss 0.7278 | f1 0.6667 | prec 0.5000 | rec 1.0000 | auc 0.4984 | ap 0.5364

Saved to movies_transfo

In [15]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

#bert
torch.cuda.empty_cache()
gc.collect()

MAX_LENGTH = 128            
BATCH_SIZE = 8       
LEARNING_RATE = 2e-6
FREEZE_TRANSFORMER = True # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "bert-base-cased"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


<>:203: SyntaxWarning: invalid escape sequence '\S'
<>:203: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_29181/3611544217.py:203: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


GPU memory cleared. Allocated: 1318MB, Cached: 1346MB
Using device: cuda
GPU memory available: 6.03 GB
Tokenizing train
Tokenizing validation
Tokenizing test

Model used (bert-base-cased)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7508.15it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

Base bert frozen. Classifier head is still trainable.
Training epochs :  20

Epoch 1/20
Model improved
train loss 0.7085 val loss 0.7007 | train f1 0.6456 val f1 0.6611 | train auc 0.4954 val auc 0.5168

Epoch 2/20
patience 1/5
train loss 0.7012 val loss 0.7001 | train f1 0.6336 val f1 0.6611 | train auc 0.5302 val auc 0.5172

Epoch 3/20
Model improved
train loss 0.7002 val loss 0.6984 | train f1 0.6379 val f1 0.6625 | train auc 0.5249 val auc 0.5175

Epoch 4/20
patience 1/5
train loss 0.7003 val loss 0.6958 | train f1 0.6201 val f1 0.6610 | train auc 0.5149 val auc 0.5179

Epoch 5/20
Model improved
train loss 0.7039 val loss 0.6944 | train f1 0.6074 val f1 0.6681 | train auc 0.4927 val auc 0.5196

Epoch 6/20
patience 1/5
train loss 0.6975 val loss 0.6937 | train f1 0.5812 val f1 0.6667 | train auc 0.5093 val auc 0.5207

Epoch 7/20
patience 2/5
train loss 0.6955 val loss 0.6934 | train f1 0.6004 val f1 0.6652 | train auc 0.5275 val auc 0.5218

Epoch 8/20
patience 3/5
train loss 0.7001 

In [16]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()


GPU memory cleared. Allocated: 1318MB, Cached: 1535MB


254